# 03 — Build Your Own ThreadPoolExecutor

**Goal:** Build a working thread pool from scratch.

## How a thread pool works

```
┌──────────────────────────────────────────────────┐
│              ThreadPool(max_workers=3)            │
│                                                  │
│  Task Queue: [task_4, task_5, task_6, ...]       │
│                                                  │
│  Workers:                                        │
│    Thread-1: running task_1                      │
│    Thread-2: running task_2                      │
│    Thread-3: running task_3                      │
│                                                  │
│  When Thread-1 finishes task_1, it picks task_4  │
│  from the queue and starts working on it.        │
└──────────────────────────────────────────────────┘
```

Key components:
1. A `queue.Queue()` of tasks (functions + args)
2. A fixed number of daemon worker threads
3. Workers loop: `queue.get()` → execute → repeat
4. `None` sentinel to signal shutdown

## Step 1 — Simple thread pool (no Future)

Build `SimpleThreadPool(max_workers)` with:
- `__init__` — creates `queue.Queue()` and starts N daemon worker threads
- `_worker()` — loop: `get()` from queue, if `None` break, else call the function
- `submit(func, *args)` — puts `(func, args)` on the queue
- `shutdown()` — `queue.join()` then send N `None` sentinels

Test: submit 10 "downloads" (each sleeps 1s). With 3 workers, total should be ~4s.

In [ ]:
import threading
import queue
import time

# Step 1 — Build SimpleThreadPool


## Step 2 — Add Future support

The real `ThreadPoolExecutor` returns `Future` objects so you can get results back.

Build `MyFuture` class:
- `_result`, `_exception` — stored when complete
- `_done = threading.Event()` — signals when result is ready
- `set_result(value)` — stores value, sets event
- `set_exception(exc)` — stores exception, sets event
- `result(timeout=None)` — waits for event, then returns result or raises exception
- `done()` — returns whether the event is set

Modify your pool: `submit()` creates a `MyFuture`, worker calls `set_result()` or `set_exception()`, `submit()` returns the future.

Test: submit 6 tasks, collect futures, call `.result()` on each.

In [ ]:
# Step 2 — Build MyFuture + MyThreadPool


## Mastery Test

Extend `MyThreadPool` with:
1. `map(func, iterable)` — submits all, returns results in order
2. Context manager: `with MyThreadPool(3) as pool: ...`
3. Compare behavior with `concurrent.futures.ThreadPoolExecutor`

They should behave identically.

In [ ]:
# Mastery Test — your code here


## Summary

You built:
- Worker threads pulling from a shared Queue
- Future objects for getting results back
- Graceful shutdown with sentinel values

This is EXACTLY how `concurrent.futures.ThreadPoolExecutor` works internally.

**Next module:** Module 03 — Multiprocessing (escape the GIL for CPU-bound work)